In [18]:
import torch
from transformers import AutoModelForSequenceClassification, logging
import time

logging.set_verbosity_error() # this just clears our cell output of some clutter

In [19]:
is_gpu_available = torch.cuda.is_available()

num_of_devices = torch.cuda.device_count()

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name()

print("Is CUDA available? ", is_gpu_available)
print("Number of CUDA devices: ", num_of_devices)
print("CUDA device name: ", device_name)

Is CUDA available?  True
Number of CUDA devices:  1
CUDA device name:  NVIDIA GeForce GTX 1660 Ti


In [20]:
# device should be 'cuda' if GPU is available, else cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# move this tensor to the available device
tensor = torch.rand(2, 2).to(device)

model = AutoModelForSequenceClassification.from_pretrained("prajjwal1/bert-tiny", num_labels=2)
model.to(device)

print("Device: ", device)
print("Tensor: ", tensor)
print("Model device: ", next(model.parameters()).device)

Device:  cuda
Tensor:  tensor([[0.5741, 0.0751],
        [0.3584, 0.5610]], device='cuda:0')
Model device:  cuda:0


In [21]:
import random
import torch
import pandas as pd
from datasets import Dataset
import peft
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)

set_seed()

In [22]:
mistral7b = 'mistralai/Mistral-7B-v0.1'
model_name = mistral7b

In [23]:
df = pd.read_csv("frankenstein_chunks.csv")

df.head()


,text
0,﻿The Project Gutenberg eBook of Frankenstein; ...
1,Further corrections by Menno de Leeuw.\n\n\n**...
2,"I am already far north of London, and as I wal..."
3,Its productions and features may be without ex...
4,But supposing all these conjectures to be fals...


In [24]:
print("Dataframe Info:")
print(df.info())
print("\n")
print("Dataframe Description:")
print(df.describe())
print("\n")
print("Number of unique values in each column:")
print(df.nunique())
random_index= random.randint(0, len(df) - 1)
df.loc[random_index, 'text']

Dataframe Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 481 entries, 0 to 480
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    481 non-null    object
dtypes: object(1)
memory usage: 3.9+ KB
None


Dataframe Description:
                                                     text
count                                                 481
unique                                                481
top     International donations are gratefully accepte...
freq                                                    1


Number of unique values in each column:
text    481
dtype: int64


'The thatch had fallen in, the walls were unplastered, and the\ndoor was off its hinges. I ordered it to be repaired, bought some\nfurniture, and took possession, an incident which would doubtless have\noccasioned some surprise had not all the senses of the cottagers been\nbenumbed by want and squalid poverty. As it was, I lived ungazed at\nand unmolested, hardly thanked for the pittance of food and clothes\nwhich I gave, so much does suffering blunt even the coarsest sensations\nof men.\n\nIn this retreat I devoted the morning to labour; but in the evening,\nwhen the weather permitted, I walked on the stony beach of the sea to\nlisten to the waves as they roared and dashed at my feet. It was a\nmonotonous yet ever-changing scene. I thought of Switzerland; it was\nfar different from this desolate and appalling landscape. '

In [25]:
df.isnull().sum()

text    0
dtype: int64

In [26]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


In [27]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config)
print("\n\nModel is running on:" + "\n")
model.device

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]



Model is running on:



device(type='cuda', index=0)

In [28]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model = prepare_model_for_kbit_training(model)
config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenized_train_dataset= train_dataset.map(lambda samples: tokenizer(samples["text"]), batched=True)
tokenized_test_dataset = test_dataset.map(lambda samples: tokenizer(samples["text"]), batched=True)

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

Map:   0%|          | 0/97 [00:00<?, ? examples/s]

In [29]:
def generate_text(prompt):
  device = "cuda"
  inputs = tokenizer(prompt, return_tensors="pt").to(device)
  outputs = model.generate(**inputs, max_new_tokens=100)
  output = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return output

In [30]:
base_generation = generate_text("I'm afraid I've created a ")
base_generation

"I'm afraid I've created a 2000-level problem with a 100-level solution.\n\nI'm a 2000-level problem.\n\nI'm a 2000-level problem.\n\nI'm a 2000-level problem.\n\nI'm a 2000-level problem.\n\nI'm a 2000-level problem.\n\nI'm a 2"

In [36]:
def calc_perplexity(model, test_dataset, tokenizer, device):
    total_perplexity = 0
    for row in test_dataset:
        inputs = tokenizer(row['text'], return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():  # منع تحديث أوزان الموديل أثناء التقييم
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            perplexity = torch.exp(loss)  # تحويل الخسارة إلى Perplexity
            total_perplexity += perplexity

    num_test_rows = len(test_dataset)
    avg_perplexity = total_perplexity / num_test_rows
    return avg_perplexity

device = next(model.parameters()).device
base_ppl = calc_perplexity(model, test_dataset, tokenizer, device)
print("Average Perplexity:", base_ppl)


tensor(0.0680, device='cuda:0')

In [37]:
import transformers

tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    args=transformers.TrainingArguments(
        warmup_steps=2,
        fp16=True,
        logging_steps=1,
        save_steps=200,
        output_dir="outputs",
      # STEP 7. Configure the training arguments.
        per_device_train_batch_size=2,
        num_train_epochs=2,
        learning_rate=2e-5,
        optim="paged_adamw_8bit"
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
# STEP 8. Finetune the model.
trainer.train()

C:\Users\Admin\anaconda3\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': 2.1851, 'grad_norm': 4.000896453857422, 'learning_rate': 0.0, 'epoch': 0.005208333333333333}
{'loss': 2.2153, 'grad_norm': 5.517046928405762, 'learning_rate': 1e-05, 'epoch': 0.010416666666666666}
{'loss': 2.3808, 'grad_norm': 7.319643974304199, 'learning_rate': 2e-05, 'epoch': 0.015625}
{'loss': 1.9169, 'grad_norm': nan, 'learning_rate': 1.9947643979057594e-05, 'epoch': 0.020833333333333332}
{'loss': 2.2297, 'grad_norm': 7.0201263427734375, 'learning_rate': 1.9947643979057594e-05, 'epoch': 0.026041666666666668}
{'loss': 2.2499, 'grad_norm': 7.564561367034912, 'learning_rate': 1.9895287958115186e-05, 'epoch': 0.03125}
{'loss': 1.6375, 'grad_norm': 8.543501853942871, 'learning_rate': 1.9842931937172775e-05, 'epoch': 0.036458333333333336}
{'loss': 2.2479, 'grad_norm': 4.670285701751709, 'learning_rate': 1.9790575916230367e-05, 'epoch': 0.041666666666666664}
{'loss': 2.1699, 'grad_norm': 4.778820514678955, 'learning_rate': 1.973821989528796e-05, 'epoch': 0.046875}
{'loss': 1.8915

C:\Users\Admin\anaconda3\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': 1.874, 'grad_norm': 4.873987197875977, 'learning_rate': 9.68586387434555e-06, 'epoch': 1.046875}
{'loss': 2.1315, 'grad_norm': 4.615110874176025, 'learning_rate': 9.633507853403143e-06, 'epoch': 1.0520833333333333}
{'loss': 1.5468, 'grad_norm': 6.042128562927246, 'learning_rate': 9.581151832460733e-06, 'epoch': 1.0572916666666667}
{'loss': 2.0595, 'grad_norm': 6.602886199951172, 'learning_rate': 9.528795811518325e-06, 'epoch': 1.0625}
{'loss': 1.0223, 'grad_norm': 5.643304347991943, 'learning_rate': 9.476439790575916e-06, 'epoch': 1.0677083333333333}
{'loss': 1.7054, 'grad_norm': 6.4011383056640625, 'learning_rate': 9.424083769633508e-06, 'epoch': 1.0729166666666667}
{'loss': 1.7732, 'grad_norm': 5.755882740020752, 'learning_rate': 9.3717277486911e-06, 'epoch': 1.078125}
{'loss': 1.8841, 'grad_norm': 5.334133625030518, 'learning_rate': 9.319371727748692e-06, 'epoch': 1.0833333333333333}
{'loss': 0.7829, 'grad_norm': 5.342809677124023, 'learning_rate': 9.267015706806284e-06, 'e

TrainOutput(global_step=384, training_loss=1.814775050074483, metrics={'train_runtime': 16372.2986, 'train_samples_per_second': 0.047, 'train_steps_per_second': 0.023, 'train_loss': 1.814775050074483, 'epoch': 2.0})

In [38]:
# STEP 9. Generate a completion with the finetuned model and compare it to the base generation.
ft_generation = generate_text("I'm afraid I've created a ")

print("Base model generation: " + base_generation + "\n\n")
print("Finetuned generation: " + ft_generation)

C:\Users\Admin\anaconda3\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\Admin\anaconda3\Lib\site-packages\torch\utils\checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Base model generation: I'm afraid I've created a 2000-level problem with a 100-level solution.

I'm a 2000-level problem.

I'm a 2000-level problem.

I'm a 2000-level problem.

I'm a 2000-level problem.

I'm a 2000-level problem.

I'm a 2


Finetuned generation: I'm afraid I've created a  monster, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekanen, #
 bekan


In [39]:
ft_ppl = calc_perplexity(model)
print("Base model perplexity: " + str(base_ppl))
print("Finetuned model perplexity: " + str(ft_ppl))

Base model perplexity: tensor(0.0680, device='cuda:0')
Finetuned model perplexity: tensor(0.0457, device='cuda:0')
